# Resident-Handling Injuries in US Nursing Facilities: What OSHA's 2025 Case Data Shows

Reproduces every figure in `README.md` from OSHA's Injury Tracking Application (ITA) case detail data.

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC)
**Data year** Calendar year 2025 (ITA submissions through 2026-08-31)
**Industry** NAICS 6231, Nursing Care Facilities (Skilled Nursing Facilities)

All source data is US Government work under 17 U.S.C. §105.

### Disclosure

The author distributes a powered patient transfer system to US healthcare facilities. That is how the author came to this data. No product is named, evaluated, or recommended here, and no claim is made that any device affects the figures described.

### What this analysis does not do

- It does not name any establishment. All outputs are national aggregates.
- It does not imply fault. OSHA states that recording an injury does not mean the employer was at fault or violated any rule.
- It does not estimate cause at the facility level. Section 9 records the hypotheses that the data did not support.

### Classification rule

"Handling injury" is not an OSHA field. It is assigned here by a keyword rule applied to the four narrative fields. The rule is printed in full in Section 5 and exported with the results, so any reader can inspect, change, and re-run it.

## 1. Environment

In [1]:
!pip install -q duckdb pandas requests

import os, re, duckdb, requests, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
con = duckdb.connect(':memory:')
def q(sql): return con.execute(sql).df()
print('duckdb', duckdb.__version__)

duckdb 1.3.2


## 2. Download

OSHA publishes ITA files at osha.gov/itadata. The file name changes with each release, and OSHA removes the earlier file when it republishes, so the URL below must be updated from that page.

This version uses the release covering submissions through 2026-08-31. Version 1.x of this repository used the earlier release through 2026-03-15, which OSHA has since replaced; figures differ between the two because the later release includes late submissions.

OSHA's server returns HTTP 403 to requests without a browser User-Agent header.

In [2]:
CASES_URL = 'https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_08-31-2026.csv'
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
PATH = 'data/osha_cases.csv'

if not os.path.exists(PATH):
    r = requests.get(CASES_URL, headers=HDRS, timeout=1800, stream=True)
    r.raise_for_status()
    with open(PATH, 'wb') as f:
        for chunk in r.iter_content(1 << 20): f.write(chunk)
print(f'{os.path.getsize(PATH)/1e6:.1f} MB')

474.8 MB


## 3. Load and inspect columns

Read verbatim as text. Column names are resolved case-insensitively below, because OSHA has renamed fields between releases.

In [3]:
con.execute(f'''CREATE OR REPLACE TABLE raw_cases AS
    SELECT * FROM read_csv_auto('{PATH}', header=true, all_varchar=true,
                                ignore_errors=true, sample_size=-1)''')
cols = [r[0] for r in con.execute('DESCRIBE raw_cases').fetchall()]
print(con.execute('SELECT COUNT(*) FROM raw_cases').fetchone()[0], 'rows')

def col(*cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    raise KeyError(f'none of {cands} found')

C = dict(
    naics    = col('naics_code'),
    job      = col('job_description'),
    soc      = col('soc_description', 'SOC_Description'),
    outcome  = col('incident_outcome'),
    dafw     = col('dafw_num_away'),
    djtr     = col('djtr_num_tr'),
    t_start  = col('time_started_work'),
    t_inc    = col('time_of_incident'),
    before   = col('new_nar_before_incident', 'nar_before_incident'),
    what     = col('new_nar_what_happened', 'nar_what_happened'),
    injury   = col('new_nar_injury_illness', 'nar_injury_illness'),
    obj      = col('new_nar_object_substance', 'nar_object_substance'),
)
C

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

782582 rows


{'naics': 'naics_code',
 'job': 'job_description',
 'soc': 'SOC_description',
 'outcome': 'incident_outcome',
 'dafw': 'dafw_num_away',
 'djtr': 'djtr_num_tr',
 't_start': 'time_started_work',
 't_inc': 'time_of_incident',
 'before': 'New_nar_before_incident',
 'what': 'New_nar_what_happened',
 'injury': 'New_nar_injury_illness',
 'obj': 'New_nar_object_substance'}

## 4. Nursing facilities (NAICS 6231)

Day counts use -9999 as a sentinel for missing values. Any negative value is set to null before summing.

In [4]:
con.execute(f'''
CREATE OR REPLACE TABLE nf AS
SELECT
  "{C['job']}"  AS job,
  "{C['soc']}"  AS soc,
  TRY_CAST("{C['outcome']}" AS INT) AS outcome,
  CASE WHEN TRY_CAST("{C['dafw']}" AS INT) >= 0 THEN TRY_CAST("{C['dafw']}" AS INT) END AS dafw,
  CASE WHEN TRY_CAST("{C['djtr']}" AS INT) >= 0 THEN TRY_CAST("{C['djtr']}" AS INT) END AS djtr,
  "{C['t_start']}" AS t_start,
  "{C['t_inc']}"   AS t_inc,
  LOWER(CONCAT_WS(' ', "{C['before']}", "{C['what']}", "{C['injury']}", "{C['obj']}")) AS narr,
  LOWER(COALESCE("{C['injury']}", '')) AS injury_text
FROM raw_cases
WHERE "{C['naics']}" LIKE '6231%'
''')
print(con.execute('SELECT COUNT(*) FROM nf').fetchone()[0], 'nursing-facility cases')

25860 nursing-facility cases


## 5. Classification rule

A case is **handling** when the combined narrative mentions a person being cared for *and* a movement of that person. It is **care, general** when it mentions a person being cared for but no movement term. Everything else is **other** (slips on wet floors, lifting boxes, needlesticks, and so on).

The strict rule is frozen. It is used unchanged in `resident-violence/` and `validation-oiics/`. The broad rule is reported alongside it so a reader can choose between precision and recall.

In [5]:
PERSON   = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE   = r"patient handling|resident handling|resident care transfer"
# Broad rule adds movement described in other words: moving, catching, lowering,
# helping into bed or chair, bearing a resident's weight.
MOVE_BROAD = (r"\b(?:mov(?:e|ed|es|ing)|catch\w*|caught|lower(?:ed|ing)?|"
              r"into (?:the |her |his |their )?(?:bed|chair|wheelchair|shower chair)|up in bed|"
              r"dead ?weight|weight of (?:the )?(?:resident|patient)|"
              r"(?:pull|push)\w* (?:the )?(?:resident|patient)|"
              r"pick\w* (?:him|her|them) up|pick\w* up (?:the )?(?:resident|patient))\b")

def classify(narr):
    # returns (strict, broad) category Series for lower-cased narratives
    p  = narr.str.contains(PERSON,     regex=True, na=False)
    m  = narr.str.contains(MOVEMENT,   regex=True, na=False)
    ph = narr.str.contains(PHRASE,     regex=True, na=False)
    b  = narr.str.contains(MOVE_BROAD, regex=True, na=False)
    strict = (p & m) | ph
    broad  = strict | (p & b)
    lab = lambda hit: pd.Series(np.where(hit, 'handling', np.where(p, 'care_general', 'other')), index=narr.index)
    return lab(strict), lab(broad)

nf = q('SELECT * FROM nf')
nf['category'], nf['category_broad'] = classify(nf.narr)

with open('derived/classification-rule.txt', 'w') as f:
    f.write(f'PERSON     = {PERSON}\nMOVEMENT   = {MOVEMENT}\nPHRASE     = {PHRASE}\n'
            f'MOVE_BROAD = {MOVE_BROAD}\n\n'
            'strict handling = (PERSON and MOVEMENT) or PHRASE\n'
            'broad handling  = strict or (PERSON and MOVE_BROAD)\n'
            'care_general    = PERSON and not handling\n'
            'other           = everything else\n')

summary = pd.DataFrame({'strict': nf.category.value_counts(),
                        'broad' : nf.category_broad.value_counts()})
summary = pd.concat([summary, (summary / summary.sum()).round(3).add_suffix('_share')], axis=1)
summary

,strict,broad,strict_share,broad_share
care_general,9082,7114,0.351,0.275
handling,5654,7622,0.219,0.295
other,11124,11124,0.430,0.430


### Validation sample

Twenty examples per category show what the rule does, not how often it is right. For that, the rule is scored against a labeled random sample.

**How the sample was drawn.** The rule was frozen first. Then 200 random nursing-facility cases were drawn (`random_state=2026`) from the earlier OSHA release (submissions through 2026-03-15). The file held only the narrative, not the rule's prediction, so labeling was blind to the rule. The same labeled sample is used here unchanged: precision and recall depend only on the labeled narratives, not on which release they came from.

**How the sample was labeled.** Each case was labeled `handling`, `care_general`, or `other` by an AI language model (Claude, Anthropic) under a written standard: a case is handling when the injury occurred while the employee was moving, supporting, or catching a resident's body. Sixteen boundary cases are explained in the `note` column. The labels were not independently reviewed by a human, so they are a silver standard. The out-of-sample test against OSHA's own coding in `validation-oiics/` is the stronger check.

The labels are published in `derived/validation-labeled.csv`. The second cell downloads them from this repository if they are not already present.

In [6]:
LAB = 'derived/validation-labeled.csv'
LAB_URL = ('https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/'
           'main/nursing-facility-handling/derived/validation-labeled.csv')
if not os.path.exists(LAB):
    open(LAB, 'wb').write(requests.get(LAB_URL, timeout=60).content)

v = pd.read_csv(LAB).dropna(subset=['label'])
v['category'], v['category_broad'] = classify(v.narr.astype(str).str.lower())
truth = v.label.str.strip() == 'handling'
out = {}
for rule in ['category', 'category_broad']:
    pred = v[rule] == 'handling'
    tp = int((pred & truth).sum())
    out[rule] = {'labeled': len(v), 'true handling': int(truth.sum()),
                 'precision': round(tp / max(pred.sum(), 1), 3),
                 'recall'   : round(tp / max(truth.sum(), 1), 3)}
val = pd.DataFrame(out).T
val.to_csv('derived/validation-result.csv')

# share of handling in the labeled sample, with a Wilson 95% interval
k, n_ = int(truth.sum()), len(v); p_ = k / n_; z = 1.96
centre = (p_ + z*z/(2*n_)) / (1 + z*z/n_)
half = z * np.sqrt(p_*(1-p_)/n_ + z*z/(4*n_*n_)) / (1 + z*z/n_)
print(f'labeled share handling: {p_:.3f}  95% CI [{centre-half:.3f}, {centre+half:.3f}]')
print(f'rule share in population (strict): {(nf.category == "handling").mean():.3f}')
val

labeled share handling: 0.230  95% CI [0.177, 0.293]
rule share in population (strict): 0.219


,labeled,true handling,precision,recall
category,200.0,46.0,0.846,0.717
category_broad,200.0,46.0,0.698,0.804


### Spot check

Twenty random narratives from each category. Read them. If the rule misclassifies, fix the rule, not the count.

In [7]:
for cat in ['handling', 'care_general', 'other']:
    print(f'===== {cat} =====')
    for t in nf[nf.category == cat].narr.sample(min(20, (nf.category == cat).sum()), random_state=1):
        print(' -', t[:220])
    print()

===== handling =====
 - rolling a resident to side to change brief. rolling a resident to side to change brief. employee stated she had pain to her right arm and right shoulder. muscle strain no object or substance used.
 - resident care assisting resident to sit and all weight went onto employee low back pain resident transfer
 - adjusting resident in bed strained lower back with lifting weight of resident lower back strain weight resistance body mechanics
 - assisting resident assisting in repositioning a resident felt pain right upper arm. left upper arm pain. poor body mechanics.
 - nurse was lifting a resident off the floor back into bed. stated lower back hurt after lifting. lower back. lifting resident back into bed.
 - patient was slipping out of wheelchair transferring patient back strain patient
 - resident care employee was pulling up the resident in the bed back pain none
 - working ee was helping another staff to turn a resident and when strainback area lower lumbarsacral 

## 6. Who is injured

In [8]:
h = nf[nf.category == 'handling'].copy()

def occupation(r):
    s = f"{r.soc or ''} {r.job or ''}".lower()
    if re.search(r'nursing assistant|\bcna\b|\bstna\b|\bcma\b|aide|orderl', s): return 'Nursing assistant / aide'
    if re.search(r'licensed practical|licensed vocational|\blpn\b|\blvn\b', s):  return 'LPN / LVN'
    if re.search(r'registered nurse|\brn\b', s):                                return 'Registered nurse'
    if re.search(r'therap', s):                                                 return 'Therapy'
    return 'Other / unspecified'

h['occupation'] = h.apply(occupation, axis=1)
by_occ = (h.occupation.value_counts().rename('cases').to_frame()
            .assign(share=lambda d: (d.cases / d.cases.sum()).round(3)))
by_occ

,cases,share
occupation,,
Nursing assistant / aide,4541,0.803
Other / unspecified,413,0.073
LPN / LVN,338,0.060
Registered nurse,274,0.048
Therapy,88,0.016


## 7. Injury type

In [9]:
h['back'] = h.injury_text.str.contains(r'back|lumbar|\bl[1-5]\b|spine|spinal', regex=True, na=False)
print(f"back / lumbar named in injury field: {h.back.sum():,} of {len(h):,} ({h.back.mean():.1%})")
h.injury_text.str.strip().value_counts().head(25)

back / lumbar named in injury field: 2,071 of 5,654 (36.6%)


,count
injury_text,
strain,149
back strain,133
strainsprain,119
lower back strain,96
lumbar strain,81
strained back,79
injury,77
low back strain,64
strain or tear,53


## 8. Severity: days away and restricted duty

In [10]:
sev = pd.Series({
    'handling cases'                     : len(h),
    'days away from work (sum)'          : int(h.dafw.sum()),
    'days restricted / transferred (sum)': int(h.djtr.sum()),
    'cases with zero days away'          : int((h.dafw.fillna(0) == 0).sum()),
    'cases > 90 days away'               : int((h.dafw > 90).sum()),
    'longest absence (days)'             : int(h.dafw.max()),
})
sev

,0
handling cases,5654
days away from work (sum),58178
days restricted / transferred (sum),125621
cases with zero days away,3598
cases > 90 days away,215
longest absence (days),348


In [11]:
hb = nf[nf.category_broad == 'handling']
pd.DataFrame({'strict': [len(h), int(h.dafw.sum()), int(h.djtr.sum()), int((h.dafw > 90).sum())],
              'broad' : [len(hb), int(hb.dafw.sum()), int(hb.djtr.sum()), int((hb.dafw > 90).sum())]},
             index=['cases', 'days away', 'days restricted', 'cases > 90 days away'])

,strict,broad
cases,5654,7622
days away,58178,79573
days restricted,125621,161604
cases > 90 days away,215,294


In [12]:
bins   = [-1, 0, 7, 30, 90, 10_000]
labels = ['0', '1-7', '8-30', '31-90', '>90']
dist = (pd.cut(h.dafw.fillna(0), bins=bins, labels=labels).value_counts()
          .reindex(labels).rename('cases').to_frame()
          .assign(share=lambda d: (d.cases / d.cases.sum()).round(3)))
dist

,cases,share
dafw,,
0,3598,0.636
1-7,1088,0.192
8-30,504,0.089
31-90,249,0.044
>90,215,0.038


## 9. Shift timing, and hypotheses that did not hold

**Night versus day.** Shift is assigned from `time_started_work`. A shift starting from 18:00 through 03:59 is counted as night. This boundary is a choice; change `NIGHT_HOURS` to test another. The difference between night and day is small and the night sample is small. No claim is made.

In [13]:
def hour(s):
    m = re.match(r'\s*(\d{1,2}):(\d{2})', str(s or ''))
    return int(m.group(1)) if m else np.nan

NIGHT_HOURS = set(range(18, 24)) | set(range(0, 4))
h['start_h'] = h.t_start.map(hour)
h['inc_h']   = h.t_inc.map(hour)
h['shift']   = np.where(h.start_h.isna(), 'unknown',
               np.where(h.start_h.isin(NIGHT_HOURS), 'night', 'day'))

by_shift = (h[h.dafw > 0].groupby('shift').dafw
              .agg(cases='count', mean_days_away='mean', median_days_away='median').round(1))
by_shift

,cases,mean_days_away,median_days_away
shift,,,
day,1525,27.5,6.0
night,304,32.3,8.0
unknown,227,28.5,6.0


**Not supported: handling injuries accumulate as a shift goes on.** If fatigue drove these injuries, the share of injuries per hour worked would rise through the shift. The table below shows the share by hour into the shift. Read it before citing fatigue.

In [14]:
h['hours_in'] = ((h.inc_h - h.start_h) % 24)
hour_in = (h.hours_in.dropna().astype(int).value_counts(normalize=True)
             .sort_index().rename('share').round(3).to_frame())
hour_in.head(13)

,share
hours_in,
0,0.059
1,0.106
2,0.111
3,0.122
4,0.120
5,0.126
6,0.097
7,0.093
8,0.044


**Not supported: facilities with higher pressure-ulcer rates have higher staff injury rates.** OSHA data carries no CMS Certification Number, so this test requires linking the two systems. It is run in [`../crosswalk/`](../crosswalk/), which links 7,757 OSHA establishments to CMS-certified nursing homes. Across 7,370 linked facilities, the DART injury rate and the long-stay pressure-ulcer rate are uncorrelated (Pearson r = −0.001; Spearman ρ = 0.024). An earlier, narrower linkage gave r = 0.011 (n = 3,497).

The likely reason is measurement, not absence of a relationship: a few handling cases per facility per year is too few to separate real differences from differences in reporting behavior. This concerns facility-level correlation only. It does not bear on the individual cases described in Sections 6 to 8.

## 10. Export

In [15]:
summary.to_csv('derived/category-summary.csv')
by_occ.to_csv('derived/handling-by-occupation.csv')
sev.rename('value').to_csv('derived/handling-severity.csv')
dist.to_csv('derived/handling-days-away-distribution.csv')
by_shift.to_csv('derived/handling-by-shift.csv')
hour_in.to_csv('derived/handling-by-hour-into-shift.csv')
for f in sorted(os.listdir('derived')): print(' ', f)

  category-summary.csv
  classification-rule.txt
  handling-by-hour-into-shift.csv
  handling-by-occupation.csv
  handling-by-shift.csv
  handling-days-away-distribution.csv
  handling-severity.csv
  validation-labeled.csv
  validation-result.csv


## Limitations

**Coverage.** ITA submission is required only of establishments meeting OSHA's size and industry thresholds. Small facilities are under-represented.

**Employer-reported.** Narratives are free text written by employers. Detail and wording vary widely.

**Rule-based classification.** "Handling" is assigned by the keyword rule in Section 5, not by OSHA. Its precision and recall against the AI-labeled random sample are reported in `derived/validation-result.csv`, and against OSHA's own coding in `../validation-oiics/`.

**Days are capped and censored.** Cases still open at submission undercount final days. Day counts use -9999 for missing and are nulled here.

**No establishment is named**, and no inference about any employer's conduct should be drawn. OSHA: recording an injury does not mean the employer was at fault or violated any OSHA rule.